> Historical course analysis. Open **modelling.ipynb** for the current ensemble comparison, selected-model explanations and deployment walkthrough. Saved outputs were cleared before public sharing. The original results.zip stays local.

# Applied ML patient modelling

This is the complete project: data contract, preprocessing, model grids, training, clustering, diagnostics, verification and the optional Streamlit dashboard. The accompanying **results.zip** holds the existing trained models and generated results. The original CSV remains one directory above this notebook.

**Run all cells** to inspect saved results and verify the project without retraining. To retrain, set `RUN_TRAINING = True` in the training cell; this replaces results.zip after the run completes. To launch the dashboard, use the optional final cell or the existing `run_dashboard.ps1` in the parent directory.

Select the Python environment at `../.venv/Scripts/python.exe`. Dependencies and exact versions are embedded below. Model archives contain patient-level data and must stay local. Only load archives created by this project.

Final code meanings, sentinel 99 handling, timing and platelet units remain provisional pending the publisher codebook. The displayed probabilities are academic source-code predictions, not clinical diagnoses.


## Environment
The existing virtual environment already contains the modelling packages. The following optional cell shows how to install dependencies in a new notebook kernel. Leave installation disabled when using the provided environment.

In [ ]:
REQUIREMENTS = 'pandas>=2.2,<4\nnumpy>=1.26,<3\nscikit-learn>=1.6,<1.11\nxgboost>=2.1,<4\nstreamlit>=1.40,<2\nmatplotlib>=3.9,<4\njoblib>=1.4,<2\nlightgbm>=4,<5\ncatboost>=1.2,<2\nthreadpoolctl>=3,<4\n'
RECORDED_ENVIRONMENT = 'altair==6.3.0\nanyio==4.15.1\nattrs==26.1.0\ncatboost==1.2.10\ncertifi==2026.7.22\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ncontourpy==1.4.0\ncycler==0.12.1\nfonttools==4.66.1\ngraphviz==0.21\nh11==0.16.0\nhttptools==0.8.0\nidna==3.20\nitsdangerous==2.2.0\nJinja2==3.1.6\njoblib==1.6.0\njsonschema-specifications==2025.9.1\njsonschema==4.26.0\nkiwisolver==1.5.1\nlightgbm==4.7.0\nMarkupSafe==3.0.4\nmatplotlib==3.11.2\nnarwhals==2.26.0\nnumpy==2.5.3\npackaging==26.3\npandas==3.0.6\npillow==12.3.0\nplotly==7.1.0\nprotobuf==7.36.2\npyarrow==25.0.1\npydeck==0.9.3\npyparsing==3.3.3\npypdf==6.19.0\npython-dateutil==2.9.0.post0\npython-multipart==0.0.32\nreferencing==0.37.0\nrequests==2.34.2\nrpds-py==2026.9.1\nscikit-learn==1.9.1\nscipy==1.18.1\nsix==1.17.0\nstarlette==1.7.0\nstreamlit==1.65.0\nthreadpoolctl==3.7.0\ntoml==0.10.2\ntyping_extensions==4.16.0\ntzdata==2026.5\nurllib3==2.8.0\nuvicorn==0.54.0\nwatchdog==6.0.0\nwebsockets==17.2\nxgboost==3.4.1\n'
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess, sys, importlib.util
    if importlib.util.find_spec('pip') is None:
        subprocess.check_call([sys.executable, '-m', 'ensurepip', '--upgrade'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *REQUIREMENTS.splitlines(), 'ipykernel'])


## Project paths and saved results
Outputs are extracted to a temporary working directory. Only the notebook and one results archive are kept in the project folder.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import os, sys, tempfile, json, shutil, subprocess, types

def find_project():
    candidates = [Path(os.environ['LEPTO_PROJECT'])] if 'LEPTO_PROJECT' in os.environ else []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([parent, parent / 'leptospirosis_project'])
    for candidate in candidates:
        if (candidate / 'analysis.ipynb').is_file() and (candidate.parent / 'Leptospirosis clinical data.csv').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Open this notebook from leptospirosis_project or set LEPTO_PROJECT to that folder.')

PROJECT = find_project()
NOTEBOOK = PROJECT / 'analysis.ipynb'
ARCHIVE = PROJECT / 'results.zip'
DATA = PROJECT.parent / 'Leptospirosis clinical data.csv'
# Loose outputs are temporary; the project keeps only this notebook and its results archive.
ROOT = Path(os.environ['LEPTO_RUNTIME_ROOT']) if 'LEPTO_RUNTIME_ROOT' in os.environ else Path(tempfile.mkdtemp(prefix='leptospirosis-'))
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
archive_stamp = str(ARCHIVE.stat().st_mtime_ns) + ':' + str(ARCHIVE.stat().st_size) if ARCHIVE.exists() else ''
stamp_file = ROOT / '.archive-stamp'
if ARCHIVE.exists() and (not stamp_file.exists() or stamp_file.read_text() != archive_stamp):
    with ZipFile(ARCHIVE) as archive:
        for entry in archive.infolist():
            destination = (ROOT / entry.filename).resolve()
            if not destination.is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe archive member: '+entry.filename)
        archive.extractall(ROOT)
    stamp_file.write_text(archive_stamp)
print('Project:', PROJECT)
print('Source data:', DATA.name)


## Data contract
Admission-only allowlist, missing-value policy, cohort eligibility and neutral target mapping.

In [ ]:
"""Data contract shared by training, dashboard and tests."""
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
SEED = 42
SYMPTOMS = ['Feverad', 'Headachead', 'Chillsad', 'Rigorsad', 'Musclepainad', 'Muscletendernessad', 'Nauseaad', 'Vomitingadmission', 'Cnsuffusionad', 'Skinrashad', 'Mucosalrashad', 'Prostrationad', 'Diarrhoeaad', 'OliguriaAd', 'Anuriaad', 'Jaundicead', 'Hepatictendernessad', 'Hepatomegalyad', 'Spleenimegalyad', 'Lympadenopathyad', 'Photophobiaad', 'Neckstiffnessad', 'Psychoticsymptomsad', 'Confusionad', 'Coughad', 'Haemoptasisad', 'SOBadd', 'Chestpainad', 'Bleedingad']
EXPOSURES = ['Marshlandexposure', 'Wetsoilexposure', 'Floodexposure', 'Forestexposure', 'BushesExposure', 'Otheragriexposure', 'Gardeningexposure', 'Otheroutdoorworkexposure', 'Naturalrunningwaterexposure', 'Stagnantwaterexposure', 'Manmaderunningwaterexposure', 'Drainsexposure', 'Otherwaterexposure', 'Paddyfieldexposure', 'CattleHandle', 'BuffaloHandle', 'Goathandle', 'Pighandle', 'Cathandle', 'Doghandle']
NUMERIC = ['Age', 'PRad', 'SBPadd', 'DBPadd']
CATEGORICAL = ['Sex'] + SYMPTOMS + EXPOSURES
FEATURES = NUMERIC + CATEGORICAL
TARGET = 'Final'
REGRESSION_TARGET = 'Platelet2'
NOTICE = 'Provisional data contract: Final 1/2 meanings and the 99 missing-value convention await the publisher CSV codebook. Class 1 means source code 1, not a verified clinical diagnosis. Research demonstration only.'

def load_data(path=DATA):
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    if raw.columns.duplicated().any():
        raise ValueError('Duplicate CSV column names.')
    required = ['Serial', TARGET, REGRESSION_TARGET] + FEATURES
    missing = set(required) - set(raw.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    if raw.Serial.str.strip().duplicated().any():
        raise ValueError('Serial must identify one patient per row; resolve duplicates before splitting.')
    X = pd.DataFrame(index=raw.index)
    for col in FEATURES:
        values = pd.to_numeric(raw[col].str.strip(), errors='coerce').replace(99, np.nan)
        if col in CATEGORICAL:
            values = values.where(values.isin([1, 2]))
            X[col] = values.map(lambda v: str(int(v)) if pd.notna(v) else np.nan).astype(object)
        else:
            X[col] = values
    bounds = {'Age': (0, 110), 'PRad': (20, 250), 'SBPadd': (40, 300), 'DBPadd': (20, 200)}
    for col, (lo, hi) in bounds.items():
        X[col] = X[col].where(X[col].between(lo, hi))
    outcome = pd.to_numeric(raw[TARGET], errors='coerce')
    observed = X[SYMPTOMS].notna().any(axis=1)
    eligible = outcome.isin([1, 2]) & observed
    y = (outcome == 1).astype(int)
    reg = pd.to_numeric(raw[REGRESSION_TARGET], errors='coerce').replace(99, np.nan)
    reg = reg.where(reg > 0)
    return (raw, X, y, eligible, reg)

def data_hash(path=DATA):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


## Course model definitions
Classification candidates, regression candidates and continuous-variable polynomial expansion.

In [ ]:
"""Predeclared course experiments. No model choices depend on test outcomes."""
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

def classification_candidates():
    """Small, explicit grids keep a full course comparison practical on a laptop."""
    balanced = dict(class_weight='balanced', random_state=SEED)
    candidates = {'Logistic regression': [LogisticRegression(C=c, max_iter=2000, **balanced) for c in (0.1, 1, 10)], 'Linear SVM': [SVC(kernel='linear', C=c, probability=True, **balanced) for c in (0.1, 1)], 'RBF SVM': [SVC(kernel='rbf', C=c, gamma=g, probability=True, **balanced) for c, g in [(1, 'scale'), (10, 'scale'), (1, 0.01)]], 'Decision tree (Gini)': [DecisionTreeClassifier(criterion='gini', max_depth=d, min_samples_leaf=leaf, ccp_alpha=a, **balanced) for d, leaf, a in [(3, 10, 0), (5, 10, 0), (None, 5, 0.01)]], 'Decision tree (entropy)': [DecisionTreeClassifier(criterion='entropy', max_depth=d, min_samples_leaf=10, ccp_alpha=a, **balanced) for d, a in [(3, 0), (5, 0.005)]], 'Bagging': [BaggingClassifier(estimator=DecisionTreeClassifier(min_samples_leaf=5, **balanced), n_estimators=100, max_samples=m, random_state=SEED, n_jobs=1) for m in (0.7, 1.0)], 'AdaBoost': [AdaBoostClassifier(n_estimators=100, learning_rate=r, random_state=SEED) for r in (0.1, 0.5)], 'Gradient boosting': [GradientBoostingClassifier(n_estimators=100, max_depth=d, learning_rate=0.05, min_samples_leaf=10, random_state=SEED) for d in (1, 2)], 'KNN': [KNeighborsClassifier(n_neighbors=k, weights='distance') for k in (5, 15, 31)], 'LightGBM': [LGBMClassifier(n_estimators=150, num_leaves=leaves, learning_rate=0.05, reg_lambda=5, class_weight='balanced', verbosity=-1, n_jobs=1, random_state=SEED) for leaves in (7, 15)], 'CatBoost': [CatBoostClassifier(iterations=150, depth=depth, learning_rate=0.05, l2_leaf_reg=5, auto_class_weights='Balanced', verbose=False, allow_writing_files=False, thread_count=2, random_seed=SEED) for depth in (3, 5)]}
    candidates['Soft voting'] = [VotingClassifier(estimators=[('lr', LogisticRegression(C=0.1, max_iter=2000, **balanced)), ('rf', RandomForestClassifier(n_estimators=150, min_samples_leaf=5, n_jobs=1, **balanced)), ('svm', SVC(C=1, kernel='rbf', probability=True, **balanced)), ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=1, learning_rate=0.05, random_state=SEED)), ('knn', KNeighborsClassifier(n_neighbors=15, weights='distance'))], voting='soft')]
    return candidates

def regression_candidates(prepare):
    """Target scaling makes penalties comparable and improves gradient descent."""

    def scaled(model, polynomial=False):
        steps = [('prepare', prepare())]
        if polynomial:
            steps.append(('basis', NumericPolynomialBasis()))
            steps.append(('basis_scale', StandardScaler()))
        steps.append(('model', model))
        return TransformedTargetRegressor(regressor=Pipeline(steps), transformer=StandardScaler())
    return {'Linear regression (OLS)': [scaled(LinearRegression())], 'Ridge regression': [scaled(Ridge(alpha=a)) for a in (1, 10, 100)], 'Lasso regression': [scaled(Lasso(alpha=a, max_iter=10000)) for a in (0.01, 0.1, 1)], 'Polynomial ridge (degree 2)': [scaled(Ridge(alpha=a), True) for a in (10, 100)], 'Gradient descent regression': [scaled(SGDRegressor(alpha=a, max_iter=5000, tol=1e-05, random_state=SEED, average=True)) for a in (0.01, 0.1)]}

class NumericPolynomialBasis(TransformerMixin, BaseEstimator):
    """ColumnTransformer emits the four numeric variables before missing indicators."""

    def fit(self, X, y=None):
        self.poly_ = PolynomialFeatures(degree=2, include_bias=False).fit(X[:, :4])
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        import numpy as np
        return np.column_stack([self.poly_.transform(X[:, :4]), X[:, 4:]])

_legacy_models = types.ModuleType('course_models')
_legacy_models.NumericPolynomialBasis = NumericPolynomialBasis
sys.modules['course_models'] = _legacy_models


## Diagnostics and lineage
Label-free development clustering, plots, uncertainty and reproducible run metadata.

In [ ]:
"""Label-free course clustering and local experiment lineage."""
import hashlib
import json
import platform
from datetime import datetime, timezone
from importlib.metadata import distributions
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering, DBSCAN, SpectralClustering, AffinityPropagation
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

def clustering_summary(Z, labels):
    """Do not treat DBSCAN noise as a cluster or score degenerate partitions."""
    keep = labels != -1
    unique = np.unique(labels[keep])
    score = None
    if 1 < len(unique) < keep.sum():
        score = float(silhouette_score(Z[keep], labels[keep], sample_size=min(700, int(keep.sum())), random_state=SEED))
    return {'clusters': len(unique), 'noise_fraction': float((~keep).mean()), 'silhouette': score, 'smallest_cluster': int(min(((labels == k).sum() for k in unique))) if len(unique) else 0}

def clustering_experiment(Xdev, prepare):
    Z = prepare().fit_transform(Xdev)
    pca = PCA(n_components=2, random_state=SEED).fit(Z)
    projection = pca.transform(Z)
    candidates = []
    for k in range(2, 7):
        candidates.extend([('K-Means', f'k={k}', KMeans(n_clusters=k, n_init=20, random_state=SEED)), ('MiniBatch K-Means', f'k={k}', MiniBatchKMeans(n_clusters=k, n_init=10, random_state=SEED)), ('Hierarchical (Ward)', f'k={k}', AgglomerativeClustering(n_clusters=k)), ('Gaussian mixture', f'k={k}', GaussianMixture(n_components=k, covariance_type='diag', n_init=3, reg_covar=0.01, random_state=SEED))])
    for eps in (2.0, 3.0, 4.0):
        candidates.append(('DBSCAN', f'eps={eps}, min_samples=10', DBSCAN(eps=eps, min_samples=10)))
    for k in (2, 3, 4):
        candidates.append(('Spectral clustering', f'k={k}', SpectralClustering(n_clusters=k, affinity='nearest_neighbors', n_neighbors=15, random_state=SEED)))
    candidates.append(('Affinity propagation', 'damping=0.9', AffinityPropagation(damping=0.9, max_iter=500, random_state=SEED)))
    rows, best = ([], {})
    import warnings
    for name, config, estimator in candidates:
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter('always')
            labels = estimator.fit_predict(Z)
            summary = clustering_summary(Z, labels)
        row = {'model': name, 'parameters': config, **summary, 'warnings': '; '.join(sorted(set((str(w.message) for w in captured))))}
        rows.append(row)
        if summary['silhouette'] is not None and (not captured):
            if name not in best or summary['silhouette'] > best[name][0]:
                best[name] = (summary['silhouette'], labels, config)
        print('CLUSTERING', name, config, summary, flush=True)
    pd.DataFrame(rows).to_csv(RESULTS / 'clustering_metrics.csv', index=False)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, (name, (_, labels, config)) in zip(axes.flat, best.items()):
        ax.scatter(projection[:, 0], projection[:, 1], c=labels, s=9, alpha=0.6, cmap='tab20')
        ax.set(title=f'{name}: {config}', xlabel='PC1', ylabel='PC2')
    for ax in list(axes.flat)[len(best):]:
        ax.axis('off')
    fig.suptitle('Development predictors only; PCA projection of full-space clusters')
    fig.tight_layout()
    fig.savefig(RESULTS / 'clustering.png', dpi=150)
    plt.close(fig)
    info = {'development_n': len(Xdev), 'uses_target_labels': False, 'uses_test_patients': False, 'representation': 'Median imputed/scaled numeric features and one-hot categorical codes, including missingness', 'pca_explained_variance_ratio': pca.explained_variance_ratio_.tolist(), 'silhouette_space': 'Full prepared feature space; noise excluded; up to 700 sampled points', 'selection': 'Highest valid silhouette within each family, development data only', 'limitation': 'Distances can reflect missingness and site collection practices; clusters are not diagnoses.'}
    (RESULTS / 'clustering_info.json').write_text(json.dumps(info, indent=2), encoding='utf-8')

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def plot_classification_curves(curves, prevalence):
    """Keep the shared legend outside the data when comparing many families."""
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    colors = plt.get_cmap('tab20')
    for i, (name, curve) in enumerate(curves.items()):
        color = colors(i)
        axes[0].plot(curve['fpr'], curve['tpr'], label=name, color=color)
        axes[1].step(curve['recall'], curve['precision'], where='post', color=color)
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
    axes[1].axhline(prevalence, color='gray', ls='--', label='Code 1 prevalence')
    axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title='Held-out ROC')
    axes[1].set(xlabel='Recall for source code 1', ylabel='Precision', title='Held-out precision-recall')
    handles, labels = axes[0].get_legend_handles_labels()
    extra_handles, extra_labels = axes[1].get_legend_handles_labels()
    fig.legend(handles + extra_handles, labels + extra_labels, loc='lower center', ncol=4, fontsize=9, frameon=False)
    fig.tight_layout(rect=(0, 0.19, 1, 1))
    fig.savefig(RESULTS / 'classification_curves.png', dpi=160)
    plt.close(fig)

def record_run(event='training_complete'):
    """Append aggregate run metadata; patient records stay in local bundles."""
    timestamp = datetime.now(timezone.utc).isoformat()
    previous = RESULTS / 'run_manifest.json'
    trained_at = timestamp
    if event == 'artifact_refresh' and previous.exists():
        prior = json.loads(previous.read_text(encoding='utf-8'))
        trained_at = prior.get('trained_at_utc', prior['completed_at_utc'])
    info = {'completed_at_utc': timestamp, 'trained_at_utc': trained_at, 'event': event, 'source_sha256': data_hash(), 'seed': SEED, 'python': platform.python_version(), 'code_sha256': {'notebook_code': hashlib.sha256('\n'.join(''.join(cell['source']) for cell in json.loads(NOTEBOOK.read_text(encoding='utf-8-sig'))['cells'] if cell['cell_type'] == 'code').encode()).hexdigest()}, 'artifact_sha256': {p.name: sha256(p) for p in sorted(RESULTS.iterdir()) if p.is_file() and p.name not in {'run_manifest.json', 'run_history.jsonl'}}, 'packages': {d.metadata['Name']: d.version for d in distributions()}, 'classification_selection': json.loads((RESULTS / 'model_selection.json').read_text()), 'regression_selection': json.loads((RESULTS / 'regression_selection.json').read_text()), 'status': 'Completed local academic experiment; no production model promotion'}
    (RESULTS / 'run_manifest.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
    with (RESULTS / 'run_history.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(info) + '\n')


## Archive storage and course mapping

In [ ]:
COURSE_COVERAGE = "# Applied ML course coverage\n\nThis project applies the modelling techniques in the supplied `Exam Prep/notes` folder to the existing admission-only patient dataset. It retains the original RF dashboard and RF + ANN assignment. The table identifies the source lesson and where to inspect each application. Week 6 contains no files.\n\n| Course material | Application | Evidence |\n| --- | --- | --- |\n| Week 1: ML Week 01; ML W2_W3; Linear_regression; Programming Exercise 1 | Supervised versus unsupervised learning; multiple linear regression by ordinary least squares; gradient descent with SGD; scaling; polynomial basis expansion; ridge L2 and lasso L1 penalties | `course_models.py`, regression metrics and development candidate scores |\n| Week 2: ML W4; logistic regression notebook; Home Work | Missing-value imputation, categorical encoding, missing indicators, standardized numeric inputs and tuned logistic L2 strength | Shared fold-fitted `preprocessor()`; logistic C grid |\n| Week 2: SVM_Classification_Assignment | Linear and RBF kernels; C and gamma tuning; shared stratified 80/20 split; accuracy, precision, recall, F1, ROC AUC and confusion matrices | Classification candidate grids, saved metrics/curves, dashboard model selector |\n| Week 3: Decision Tree Lab; decision trees; Information Gain/Gini lecture note | Separate entropy and Gini tree families, pre-pruning with depth/leaf limits and cost-complexity pruning with predeclared ccp_alpha | Selected tree configurations, train/validation gaps, dashboard selected tree plot |\n| Week 3: Random Forest notebook and RF slides | Bootstrap aggregation, random feature subsets, forest comparison with a single tree and bagging; permutation importance | Bagging/RF candidates; `rf_importance.csv`; RF explorer |\n| Week 4: Ensemble Learning Practice I | Logistic, RF, SVM, gradient boosting and KNN learners; a predeclared soft-voting ensemble | All five members included in `Soft voting`; shared outer folds |\n| Week 4: XGBoost Algorithm and ML XG Boost | AdaBoost, gradient boosting, XGBoost, LightGBM and CatBoost | Tuned held-out benchmark for each family |\n| Week 4: Clustering Models AI; Clustering; K Means notebook | K-Means, MiniBatch K-Means, Ward hierarchical clustering, DBSCAN, Gaussian mixtures, spectral clustering and affinity propagation; silhouette analysis and PCA projection | `course_diagnostics.py`, clustering tables and plot |\n| Week 4: Hands on Project | RF dashboard, RF versus XGBoost, RF + ANN tabular regression, comparison of ensemble methods | Original tools preserved; expanded comparisons in dashboard/report |\n| Week 5: Introduction to MLOps; workshop commands | Reusable training pipeline, data contract, automated tests, exact package versions, source/code/artifact hashes, timestamped append-only aggregate run history and development-selected model record | `data_contract.py`, tests, `requirements-lock.txt`, `run_manifest.json`, `run_history.jsonl`, selection JSON files |\n\n## Design choices\n\nAll classification families use the same five development folds and the same original held-out patient split. Candidate selection maximizes development mean ROC AUC. All regression candidates use the same three development folds and select by mean RMSE. Candidate lists are intentionally compact for local execution; they are not exhaustive searches. Train-versus-validation metrics expose overfitting. The family with the best development score is recorded before test evaluation; test results are descriptive comparisons and never determine ensemble membership or configuration. Comparing many families still creates model-selection uncertainty. Historical test results were already available in the original project, so this is an expanded internal benchmark rather than a new independent validation.\n\nMedian imputation, numeric scaling, one-hot encoding and basis construction fit inside each training fold. Polynomial regression expands the four continuous admission variables to degree two, retaining encoded categorical fields. Ridge penalizes the expanded model to limit variance in the small regression cohort. SGD uses a standardized target and input scaling. Both SVM kernels use numeric scaling; one-hot categorical values are already bounded. SVC's internal probability fitting uses only the fitting fold; probabilities remain provisional and require independent calibration assessment.\n\nThe soft-voting recipe is fixed before scoring. RF + ANN stacking fits its combiner on development out-of-fold predictions with preprocessing inside each base learner. Its internal CV is nested inside each outer regression fold. ANN/stack convergence warnings are saved. CatBoost and LightGBM deliberately receive the same encoded representation for comparability; this does not test CatBoost's native categorical handling. Class weighting differs across algorithms and is recorded in selected configurations; AdaBoost, gradient boosting and KNN use their ordinary fitting objectives.\n\nClustering receives only development predictors and no outcome labels. Fit and silhouette use the complete prepared feature space; PCA is only a two-dimensional display. Silhouette excludes DBSCAN noise and is undefined for fewer than two usable clusters. Noise fraction, cluster counts, smallest cluster and warnings are reported alongside scores. A high silhouette on a small retained fraction does not establish useful segmentation. Patient missingness and collection practices may dominate distances. Clusters must not be called disease subtypes. Mixtures use diagonal covariance and regularization for the encoded feature space.\n\n## Techniques taught as context or requiring a different problem\n\n- The Gaussian basis example in the Week 1 notebook illustrates nonlinear basis regression. This project implements the same basis-expansion concept with degree-two polynomials; it does not claim a separate Gaussian-basis fit.\n- OOB evaluation is explained by bootstrap aggregation but is not reported as independent validation: the shared preprocessing is fitted before tree bootstrap sampling. The shared development folds are the comparison criterion.\n- t-SNE, image colour compression and digit recognition are illustrative notebook applications. PCA provides the requested cluster projection on patient features without implying a predictive embedding. Synthetic and unrelated dataset examples are not copied into this benchmark.\n- General introductory mentions of reinforcement learning, association rules, autoencoders, OPTICS/DenClue and constrained/grid clustering are contextual families rather than lab requirements. The implemented clustering comparisons cover the demonstrated lab models and spectral/affinity examples; disease-code prediction is not a sequential reward or transaction-basket problem.\n- RF + ARIMA needs a suitable regular time series. Patient rows with short irregular follow-up are not such a series, so RF + ANN fulfils the tabular hybrid task.\n- Week 5's DVC/DagsHub storage, MLflow registry, SHAP, FastAPI/Docker, GitHub CI/CD, Azure canary/rollback and live drift/retraining are production workshop integrations. This modelling upgrade provides local lineage and tests, and does not claim those services are configured. A production monitoring stream, labelled feedback, hosting and external accounts are absent. Local model selection is not production promotion. Permutation importance remains the implemented explanatory technique.\n\nThe publisher codebook is still required to verify Final code meanings, 99 missing-value conventions, measurement timing and platelet units. No additional algorithm resolves those uncertainties.\n\n\n## Consolidated layout\nThe former module, report and test filenames above identify their original roles. Their complete active contents now live in the corresponding sections of this notebook. Saved outputs retain their original internal names inside results.zip. Standalone maintenance scripts have been removed.\n"

def save_results():
    """Atomically replace one archive; preserve the previous archive if writing fails."""
    (ROOT / 'COURSE_COVERAGE.md').write_text(COURSE_COVERAGE, encoding='utf-8')
    staged_archive = ARCHIVE.with_suffix('.zip.tmp')
    with ZipFile(staged_archive, 'w', compression=ZIP_DEFLATED) as archive:
        for artifact in sorted(RESULTS.iterdir()):
            if artifact.is_file():
                archive.write(artifact, 'results/'+artifact.name)
        for name in ['REPORT.md', 'COURSE_COVERAGE.md']:
            if (ROOT/name).exists():
                archive.write(ROOT/name, name)
    os.replace(staged_archive, ARCHIVE)

(ROOT / 'COURSE_COVERAGE.md').write_text(COURSE_COVERAGE, encoding='utf-8')


## Shared training workflow
Every candidate fits preprocessing inside its development fold. Held-out patients do not choose model parameters, voting members or thresholds. The original historical split is an internal benchmark, not fresh independent validation.

In [ ]:
"""Course-aligned classification, regression, ensemble and clustering experiments."""
import json
import platform
import time
import warnings
from pathlib import Path
import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, roc_curve, precision_recall_curve, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits

def preprocessor():
    numeric = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)), ('scale', StandardScaler())])
    categorical = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Missing', keep_empty_features=True)), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    return ColumnTransformer([('numeric', numeric, NUMERIC), ('category', categorical, CATEGORICAL)])

def pipeline(model):
    return Pipeline([('prepare', preprocessor()), ('model', model)])

def fit_classifier(pipe, X, y):
    kwargs = {}
    if isinstance(pipe.named_steps['model'], XGBClassifier):
        kwargs['model__sample_weight'] = compute_sample_weight('balanced', y)
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        pipe.fit(X, y, **kwargs)
    pipe.fit_warnings_ = sorted(set((str(w.message) for w in captured)))
    return pipe

def classification_metrics(y, p, threshold=0.5):
    predicted = (p >= threshold).astype(int)
    return {'accuracy': accuracy_score(y, predicted), 'balanced_accuracy': balanced_accuracy_score(y, predicted), 'precision_code1': precision_score(y, predicted, zero_division=0), 'recall_code1': recall_score(y, predicted, zero_division=0), 'f1_code1': f1_score(y, predicted, zero_division=0), 'roc_auc': roc_auc_score(y, p), 'average_precision': average_precision_score(y, p), 'brier_score': brier_score_loss(y, p)}

def regression_metrics(y, p):
    return {'mae': mean_absolute_error(y, p), 'rmse': float(np.sqrt(mean_squared_error(y, p))), 'r2': r2_score(y, p)}

def bootstrap_intervals(y, p):
    rng = np.random.default_rng(SEED)
    collected = []
    y = np.asarray(y)
    for _ in range(500):
        indices = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[indices])) == 2:
            collected.append(classification_metrics(y[indices], p[indices]))
    return {k: np.quantile([r[k] for r in collected], [0.025, 0.975]).tolist() for k in collected[0]}

def save_json(name, obj):
    (RESULTS / name).write_text(json.dumps(obj, indent=2, default=lambda v: v.item()), encoding='utf-8')

def classification_experiment(X, y, raw):
    train_ids, test_ids = train_test_split(X.index, test_size=0.2, random_state=SEED, stratify=y)
    Xtr, ytr = (X.loc[train_ids], y.loc[train_ids])
    Xte, yte = (X.loc[test_ids], y.loc[test_ids])
    folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xtr, ytr))
    grids = {'Random Forest': [dict(n_estimators=250, min_samples_leaf=leaf, max_features=mf, class_weight='balanced', n_jobs=1, random_state=SEED) for leaf, mf in [(2, 'sqrt'), (5, 'sqrt'), (5, 0.5)]], 'XGBoost': [dict(n_estimators=200, max_depth=depth, learning_rate=rate, subsample=0.8, colsample_bytree=0.8, reg_lambda=5, min_child_weight=5, tree_method='hist', n_jobs=1, random_state=SEED, eval_metric='logloss') for depth, rate in [(2, 0.05), (3, 0.05), (2, 0.1)]]}
    candidates = classification_candidates()
    candidates.update({name: [RandomForestClassifier(**params) if name == 'Random Forest' else XGBClassifier(**params) for params in grid] for name, grid in grids.items()})
    fitted = {}
    cv_rows = []
    configs = {}
    for name, grid in candidates.items():
        scores = []
        for candidate, estimator in enumerate(grid):
            parameters = estimator.get_params(deep=False)
            fold_scores = []
            for fold, (a, b) in enumerate(folds):
                start = time.perf_counter()
                pipe = fit_classifier(pipeline(clone(estimator)), Xtr.iloc[a], ytr.iloc[a])
                pred = pipe.predict_proba(Xtr.iloc[b])[:, 1]
                fold_scores.append(roc_auc_score(ytr.iloc[b], pred))
                cv_rows.append({'model': name, 'candidate': candidate, 'fold': fold + 1, 'roc_auc': fold_scores[-1], 'train_roc_auc': roc_auc_score(ytr.iloc[a], pipe.predict_proba(Xtr.iloc[a])[:, 1]), 'fit_seconds': time.perf_counter() - start, 'fit_warnings': '; '.join(pipe.fit_warnings_), 'parameters': json.dumps(parameters, default=str)})
            scores.append(float(np.mean(fold_scores)))
            print(name, 'candidate', candidate, 'CV AUC', round(scores[-1], 4), flush=True)
        selected = int(np.argmax(scores))
        estimator = grid[selected]
        configs[name] = {'candidate': selected, 'parameters': json.loads(json.dumps(estimator.get_params(deep=False), default=str)), 'cv_mean_auc': scores[selected]}
        fitted[name] = fit_classifier(pipeline(clone(estimator)), Xtr, ytr)
        configs[name]['final_fit_warnings'] = fitted[name].fit_warnings_
    champion = max(configs, key=lambda name: configs[name]['cv_mean_auc'])
    save_json('model_selection.json', {'selection_metric': 'development mean ROC AUC', 'selected_model': champion, 'cv_mean_auc': configs[champion]['cv_mean_auc'], 'test_used_for_selection': False, 'seed': SEED, 'folds': len(folds)})
    clustering_experiment(Xtr, preprocessor)
    fitted['Majority baseline'] = pipeline(DummyClassifier(strategy='prior')).fit(Xtr, ytr)
    rows, intervals, curves, predictions = ([], {}, {}, {})
    for name, pipe in fitted.items():
        p = pipe.predict_proba(Xte)[:, 1]
        predictions[name] = p
        row = {'model': name, **classification_metrics(yte, p)}
        rows.append(row)
        intervals[name] = bootstrap_intervals(yte, p)
        fpr, tpr, _ = roc_curve(yte, p)
        precision, recall, _ = precision_recall_curve(yte, p)
        curves[name] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'precision': precision.tolist(), 'recall': recall.tolist(), 'confusion_matrix': confusion_matrix(yte, p >= 0.5).tolist()}
        print(name, 'TEST', row, flush=True)
    pd.DataFrame(rows).to_csv(RESULTS / 'classification_metrics.csv', index=False)
    pd.DataFrame(cv_rows).to_csv(RESULTS / 'classification_cv.csv', index=False)
    importance = permutation_importance(fitted['Random Forest'], Xte, yte, scoring='roc_auc', n_repeats=10, random_state=SEED, n_jobs=1)
    pd.DataFrame({'feature': FEATURES, 'mean_auc_drop': importance.importances_mean, 'std_auc_drop': importance.importances_std}).sort_values('mean_auc_drop', ascending=False).to_csv(RESULTS / 'rf_importance.csv', index=False)
    save_json('classification_curves.json', curves)
    save_json('classification_intervals.json', intervals)
    save_json('selected_configs.json', configs)
    joblib.dump({'models': fitted, 'X_train': Xtr, 'y_train': ytr, 'X_test': Xte, 'y_test': yte, 'probabilities': predictions, 'source_hash': data_hash(), 'selected_model': champion}, RESULTS / 'classification_bundle.joblib')
    plot_classification_curves(curves, float(yte.mean()))
    hospital = raw.loc[X.index, 'Hospital']
    site = hospital.value_counts().index[0]
    site_test = hospital == site
    site_result = {'held_out_hospital_code': str(site), 'train_n': int((~site_test).sum()), 'test_n': int(site_test.sum())}
    if y[site_test].nunique() == 2 and y[~site_test].nunique() == 2:
        for name in ['Random Forest', 'XGBoost']:
            parameters = grids[name][0]
            estimator = RandomForestClassifier(**parameters) if name == 'Random Forest' else XGBClassifier(**parameters)
            p = fit_classifier(pipeline(estimator), X[~site_test], y[~site_test]).predict_proba(X[site_test])[:, 1]
            site_result[name] = classification_metrics(y[site_test], p)
    save_json('hospital_sensitivity.json', site_result)
    return (rows, len(train_ids), len(test_ids))

def hybrid_experiment(X, target):
    ids = target.index
    train_ids, test_ids = train_test_split(ids, test_size=0.2, random_state=SEED)
    Xtr, ytr = (X.loc[train_ids], target.loc[train_ids])
    Xte, yte = (X.loc[test_ids], target.loc[test_ids])
    rf = pipeline(RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.7, random_state=SEED, n_jobs=1))
    ann = TransformedTargetRegressor(regressor=pipeline(MLPRegressor(hidden_layer_sizes=(16,), solver='lbfgs', alpha=10, max_iter=1000, random_state=SEED)), transformer=StandardScaler())
    stack = StackingRegressor(estimators=[('rf', clone(rf)), ('ann', clone(ann))], final_estimator=RidgeCV(alphas=[0.1, 1, 10, 100]), cv=KFold(3, shuffle=True, random_state=SEED), n_jobs=1)
    models = {'Mean baseline': [pipeline(DummyRegressor())], 'RF regression': [rf], 'ANN regression': [ann], 'RF + ANN stacking': [stack]}
    models.update(regression_candidates(preprocessor))
    cv = list(KFold(3, shuffle=True, random_state=SEED).split(Xtr))
    rows, cv_rows, predictions = ([], [], {})
    warning_messages = []
    fitted_models, selected_configs = ({}, {})
    for name, candidates in models.items():
        start = time.perf_counter()
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter('always')
            candidate_scores = []
            for candidate, model in enumerate(candidates):
                fold_rmse = []
                for fold, (a, b) in enumerate(cv):
                    fitted = clone(model).fit(Xtr.iloc[a], ytr.iloc[a])
                    metrics = regression_metrics(ytr.iloc[b], fitted.predict(Xtr.iloc[b]))
                    fold_rmse.append(metrics['rmse'])
                    cv_rows.append({'model': name, 'candidate': candidate, 'fold': fold + 1, 'train_rmse': regression_metrics(ytr.iloc[a], fitted.predict(Xtr.iloc[a]))['rmse'], **metrics})
                candidate_scores.append(float(np.mean(fold_rmse)))
            selected = int(np.argmin(candidate_scores))
            model = clone(candidates[selected]).fit(Xtr, ytr)
            fitted_models[name] = model
            selected_configs[name] = {'candidate': selected, 'cv_mean_rmse': candidate_scores[selected], 'estimator': str(candidates[selected])}
            warning_messages.extend((str(w.message) for w in captured))
        pred = model.predict(Xte)
        predictions[name] = pred
        rows.append({'model': name, **regression_metrics(yte, pred), 'training_seconds_including_cv': time.perf_counter() - start})
        print('REGRESSION', rows[-1], flush=True)
    save_json('regression_selected_configs.json', selected_configs)
    save_json('regression_selection.json', {'selected_model': min(selected_configs, key=lambda name: selected_configs[name]['cv_mean_rmse']), 'selection_metric': 'development mean RMSE', 'test_used_for_selection': False})
    pd.DataFrame(rows).to_csv(RESULTS / 'regression_metrics.csv', index=False)
    pd.DataFrame(cv_rows).to_csv(RESULTS / 'regression_cv.csv', index=False)
    save_json('regression_info.json', {'target': REGRESSION_TARGET, 'eligible_n': len(ids), 'train_n': len(train_ids), 'test_n': len(test_ids), 'warnings': sorted(set(warning_messages)), 'units': 'Source count units; codebook verification pending', 'timing': 'Day two inferred from column suffix; verify against CSV codebook'})
    joblib.dump({'models': fitted_models, 'actual': yte, 'predictions': predictions, 'source_hash': data_hash()}, RESULTS / 'regression_bundle.joblib')
    fig, ax = plt.subplots(figsize=(7, 5))
    for name in ['RF regression', 'RF + ANN stacking']:
        ax.scatter(yte, predictions[name], alpha=0.7, label=name)
    lo, hi = (min(yte.min(), min((p.min() for p in predictions.values()))), max(yte.max(), max((p.max() for p in predictions.values()))))
    ax.plot([lo, hi], [lo, hi], 'k--')
    ax.set(xlabel='Recorded Platelet2', ylabel='Predicted Platelet2', title='Exploratory regression: held-out patients')
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS / 'hybrid_regression.png', dpi=160)
    plt.close(fig)
    return (rows, len(ids))

@threadpool_limits.wrap(limits=2)
def main():
    RESULTS.mkdir(exist_ok=True)
    raw, X, y, eligible, reg = load_data()
    audit = {'source_file': DATA.name, 'sha256': data_hash(), 'rows': len(raw), 'columns': len(raw.columns), 'unique_patient_ids': int(raw.Serial.nunique()), 'source_final_counts': raw.Final.value_counts().to_dict(), 'classification_eligible': int(eligible.sum()), 'excluded_no_admission_symptoms': int((~eligible).sum()), 'selected_features': FEATURES, 'excluded_fields_count': len(raw.columns) - len(FEATURES) - 1, 'label_mapping': {'0': 'Source Final code 2', '1': 'Source Final code 1'}, 'codebook_verified': False, 'missing_policy': 'Blank/non-numeric and numeric 99 treated as missing for allowlisted fields; provisional.', 'notice': NOTICE, 'versions': {'python': platform.python_version(), 'pandas': pd.__version__, 'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__}}
    pd.DataFrame({'feature': FEATURES, 'missing_fraction': X[eligible].isna().mean(), 'type': ['numeric' if c in NUMERIC else 'categorical code' for c in FEATURES]}).to_csv(RESULTS / 'feature_audit.csv', index=False)
    pd.DataFrame({'column': raw.columns, 'selected_predictor': raw.columns.isin(FEATURES), 'role': ['predictor' if c in FEATURES else 'classification target' if c == TARGET else 'regression target' if c == REGRESSION_TARGET else 'excluded' for c in raw.columns]}).to_csv(RESULTS / 'column_manifest.csv', index=False)
    classification, train_n, test_n = classification_experiment(X[eligible], y[eligible], raw)
    reg_eligible = reg.notna() & eligible
    regression, reg_n = hybrid_experiment(X[reg_eligible], reg[reg_eligible])
    audit.update(classification_train_n=train_n, classification_test_n=test_n, regression_eligible=reg_n)
    save_json('data_audit.json', audit)
    record_run()
    report = make_report(audit, classification, regression)
    (ROOT / 'REPORT.md').write_text(report, encoding='utf-8')
    save_results()
    print('Completed. Saved all results to', ARCHIVE, flush=True)

def markdown_table(rows, keys):
    lines = ['| ' + ' | '.join(keys) + ' |', '| ' + ' | '.join(['---'] * len(keys)) + ' |']
    for r in rows:
        lines.append('| ' + ' | '.join((f'{r[k]:.4f}' if isinstance(r[k], float) else str(r[k]) for k in keys)) + ' |')
    return '\n'.join(lines)

def make_report(audit, classification, regression):
    selection = json.loads((RESULTS / 'model_selection.json').read_text())
    reg_selection = json.loads((RESULTS / 'regression_selection.json').read_text())
    cluster_info = json.loads((RESULTS / 'clustering_info.json').read_text())
    return f"# Applied ML patient modelling study\n\n## Scope and course coverage\n\nThe expanded benchmark implements the course's regression, classification, tree, ensemble and clustering labs. See the course coverage section in this notebook for source lessons, implementation locations, design choices and contextual techniques that require other data or production infrastructure. {NOTICE}\n\nThe original CSV is unchanged: {audit['rows']} patients, {audit['columns']} columns. Admission eligibility retains {audit['classification_eligible']} patients and {len(FEATURES)} predictors. Final code 1 is encoded as 1 and Final code 2 as 0; neither code is given an unverified diagnostic meaning.\n\n## Method and leakage controls\n\nAn explicit allowlist excludes identifiers, hospital, sample category, year/month, ICU/OPD, diagnostic tests and all follow-up measurements. Unsuffixed laboratory measurements are excluded because admission timing is uncertain. Numeric plausibility bounds and provisional 99/blank handling are defined in data_contract.py. A row requires at least one recorded admission symptom; this selects a subgroup and may encode site collection practices.\n\nThe original seed-42 stratified 80/20 classification split is preserved: {audit['classification_train_n']} development and {audit['classification_test_n']} test patients. Every non-dummy classifier uses five shared development folds. Each candidate refits its imputer, encoder and numeric scaler inside each fold. Predeclared compact grids tune logistic regularization, SVM C/gamma, tree impurity/depth/leaf/pruning, ensemble complexity and KNN neighbours. RF/logistic/SVM/trees/LightGBM/CatBoost use balanced class weights; XGBoost computes balanced sample weights from each fitting fold. Other algorithms use ordinary fitting objectives. Soft voting averages a fixed five-member recipe; its members are not chosen using test performance.\n\nSelected classification family: **{selection['selected_model']}**, development mean ROC AUC **{selection['cv_mean_auc']:.4f}**. This choice is recorded before test evaluation. The default threshold is 0.5. No test-based feature, threshold or architecture tuning occurs. The historical project already exposed this same test split; expanded comparisons are internal research, not fresh independent validation. Trying many families and selecting the largest CV mean can produce optimism; fold dispersion is not a confidence interval for the full search.\n\n## Held-out classification results\n\n{markdown_table(classification, ['model', 'accuracy', 'balanced_accuracy', 'precision_code1', 'recall_code1', 'f1_code1', 'roc_auc', 'average_precision', 'brier_score'])}\n\nPrecision, recall and F1 refer to source code 1. The saved confusion matrices use rows actual / columns predicted in order [code 2, code 1], so [[TN, FP], [FN, TP]] refers to that numerical positive class. ROC AUC measures ranking across thresholds; average precision summarizes precision/recall and depends on prevalence. Brier is lower-is-better. SVM probabilities are fitted internally using only fitting records, but none of these probabilities should be interpreted as calibrated clinical risks.\n\nclassification_cv.csv records candidate, fold, training AUC, validation AUC, fit time and parameters. Large train/validation gaps suggest overfitting. selected_configs.json records each family's selected configuration. classification_intervals.json contains 500-resample fixed-model percentile intervals; these do not cover full selection uncertainty or establish significant differences. classification_curves.json stores every model's ROC, precision-recall and confusion matrix.\n\nThe largest-hospital sensitivity remains a separate RF/XGBoost analysis using the first predeclared configurations, without selecting on held-out site records. It is not external validation. RF permutation importance is descriptive, performed after selection, and is not used to filter predictors. Correlated fields can share importance and associations are not causal.\n\n## Continuous-outcome regression\n\nThe separate Platelet2 cohort has {audit['regression_eligible']} patients with observed positive targets. Target values are never imputed. Timing (day two inferred from suffix) and source count units remain unverified. A separate original seed-42 80/20 split and three shared development folds compare mean baseline, OLS, ridge, lasso, polynomial ridge, gradient-descent SGD, RF, ANN and RF + ANN stacking. Regression candidate selection minimizes mean development RMSE. Selected family: **{reg_selection['selected_model']}**.\n\nThe four continuous admission inputs receive degree-two polynomial expansion; categories and missing indicators are retained. Ridge regularization controls the expanded model. OLS/regularized/SGD models standardize the target inside fitting folds. ANN inputs and targets are standardized. RF + ANN learns a ridge combiner from development out-of-fold base predictions; the base pipelines fit preprocessing within each stacking fold. Internal stacking CV is nested inside the outer regression folds. Full candidates and selected settings are saved in regression_selected_configs.json; train/validation RMSE is saved in regression_cv.csv.\n\n{markdown_table(regression, ['model', 'mae', 'rmse', 'r2'])}\n\nLower MAE/RMSE is better in unverified source units. Negative R2 means worse squared error than predicting this test set's mean; that reference is not a deployable baseline. The trained mean baseline is reported explicitly. regression_info.json records captured fitting/convergence warnings. The small follow-up cohort and ANN warnings constrain interpretation; adding complexity does not imply improvement.\n\n## Unsupervised development exploration\n\nClustering uses {cluster_info['development_n']} development patients, all admission predictors and no class labels or held-out patients. K-Means, MiniBatch K-Means, Ward hierarchical clustering, diagonal-covariance Gaussian mixtures, DBSCAN, spectral clustering and affinity propagation are compared. Each k-based family explores predeclared cluster counts; DBSCAN explores predeclared radii. All share the development-fitted prepared representation.\n\nclustering_metrics.csv records full-space silhouette, cluster count, noise fraction, smallest cluster and warnings. Noise is excluded from DBSCAN silhouette; scores are undefined for degenerate partitions. A high score with a large noise fraction is not evidence of broad segmentation quality. The best valid silhouette within each family is plotted, without claiming one algorithm is universally best. PCA only displays full-space clusters; the two plotted components explain {sum(cluster_info['pca_explained_variance_ratio']):.1%} of prepared feature variance. The plot may hide substantial structure. Missingness/site practices may dominate clustering; clusters are not diagnoses.\n\n## Reproducibility and MLOps\n\nA single training entry point writes aggregate metrics, model selections, scientific plots and local model bundles. run_manifest.json records UTC completion time, seed, exact packages and SHA-256 hashes of source data, code and generated experiment artifacts. run_history.jsonl appends aggregate run records. Tests check data contracts, splitting, preprocessing isolation and model outputs. The notebook source is hashed as the complete implementation; the generated report is documentation.\n\nThis is a local academic workflow. No production registry, cloud storage, service deployment, continuous monitoring or automated model promotion is configured. The MLOps workshop integrations and their prerequisites are explicitly mapped in COURSE_COVERAGE.md. Bundles contain patient-level subsets and stay local; only aggregate downloads are exposed by the dashboard.\n\n## Limitations and interpretation\n\nThe publisher codebook must verify Final meanings, sentinels, timing and platelet units before stronger interpretation. Hospital recruitment, complete-admission eligibility and observed platelet follow-up select nonrepresentative groups. Random holdout and one hospital sensitivity do not establish temporal or external validity. Class weighting and missingness can affect probabilities. Independent validation, calibration and verified labels remain required for clinical research claims. RF + ARIMA is not applied to irregular patient follow-up because it requires a suitable regular time series.\n\nThe comparison demonstrates linear versus nonlinear decision functions, regularization, variance reduction by bagging, sequential boosting and prediction combination by voting/stacking on tabular data. Image/text tasks require suitable representations; this experiment does not establish performance on those modalities.\n\n## Sources\n\n- Course materials: ../../Exam Prep/notes (mapped by file in COURSE_COVERAGE.md).\n- Original assignment: week 4/HandsonProject.jpeg.\n- Dataset, CC BY 4.0: https://data.mendeley.com/datasets/zmxmp42g9p/4\n- Data paper: https://doi.org/10.1016/j.dib.2022.108378\n- Supporting questionnaires in the workspace root; these do not replace the CSV codebook.\n"


## Load or reproduce the experiment
Set the flag to True only when you want to retrain. A full run can take several minutes. The generated archive is replaced only after training finishes.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    main()
if not (RESULTS / 'data_audit.json').exists():
    raise FileNotFoundError('No saved results. Set RUN_TRAINING = True and rerun this cell.')
if json.loads((RESULTS / 'data_audit.json').read_text())['sha256'] != data_hash():
    raise ValueError('Source data changed; retrain before interpreting saved results.')


## Data audit and classification results

In [ ]:
from IPython.display import display, Markdown, Image
display(pd.read_csv(RESULTS / 'feature_audit.csv'))
display(pd.read_csv(RESULTS / 'classification_metrics.csv'))
display(Image(filename=str(RESULTS / 'classification_curves.png')))
display(pd.read_csv(RESULTS / 'classification_cv.csv').groupby(['model','candidate'])[['train_roc_auc','roc_auc']].agg(['mean','std']))
print('Development selection:', json.loads((RESULTS / 'model_selection.json').read_text()))


## Classifier and RF exploration
Change the model name, threshold or tree index and rerun this cell. Threshold changes are descriptive and do not alter published results.

In [ ]:
bundle = joblib.load(RESULTS / 'classification_bundle.joblib')
MODEL_NAME = bundle.get('selected_model', 'Random Forest')
THRESHOLD = 0.50
TREE_INDEX = 0
y_test = bundle['y_test']
p_test = bundle['probabilities'][MODEL_NAME]
display(pd.DataFrame([classification_metrics(y_test, p_test, THRESHOLD)]))
display(pd.DataFrame(confusion_matrix(y_test,p_test >= THRESHOLD,labels=[0,1]),
    index=['Actual code 2','Actual code 1'],columns=['Predicted code 2','Predicted code 1']))
display(pd.read_csv(RESULTS / 'rf_importance.csv').head(15))
from sklearn.tree import plot_tree
forest = bundle['models']['Random Forest']
fig, ax = plt.subplots(figsize=(18,7))
plot_tree(forest.named_steps['model'].estimators_[TREE_INDEX],
    feature_names=forest.named_steps['prepare'].get_feature_names_out(),
    class_names=['Code 2','Code 1'],max_depth=2,filled=True,fontsize=8,ax=ax)
display(fig)
plt.close(fig)


## Hypothetical source-code prediction
Edit source-coded values in the example. At least one admission symptom is required, just as in the training cohort. Numeric bounds and sentinel handling match the data contract.

In [ ]:
def predict_record(values, model_name='Random Forest'):
    unexpected = set(values) - set(FEATURES)
    if unexpected:
        raise ValueError('Unknown predictors: '+str(sorted(unexpected)))
    row = {c: np.nan for c in FEATURES}
    bounds = {'Age':(0,110),'PRad':(20,250),'SBPadd':(40,300),'DBPadd':(20,200)}
    for column, value in values.items():
        numeric = pd.to_numeric(value, errors='coerce')
        if pd.isna(numeric) or numeric == 99:
            continue
        if column in CATEGORICAL:
            row[column] = str(int(numeric)) if numeric in (1,2) else np.nan
        else:
            lo, hi = bounds[column]
            row[column] = float(numeric) if lo <= numeric <= hi else np.nan
    if all(pd.isna(row[c]) for c in SYMPTOMS):
        raise ValueError('Enter at least one admission symptom code.')
    return float(bundle['models'][model_name].predict_proba(pd.DataFrame([row])[FEATURES])[0,1])

EXAMPLE_RECORD = {'Age':35, 'Sex':1, 'Feverad':1}
print('Uncalibrated probability of Final code 1:', predict_record(EXAMPLE_RECORD))


## Regression results
Target: recorded Platelet2. Units and timing remain provisional. Inspect captured warnings before interpreting ANN or hybrid performance.

In [ ]:
display(pd.read_csv(RESULTS / 'regression_metrics.csv'))
display(pd.read_csv(RESULTS / 'regression_cv.csv').groupby(['model','candidate'])[['train_rmse','rmse']].mean())
display(Image(filename=str(RESULTS / 'hybrid_regression.png')))
print('Development selection:', json.loads((RESULTS / 'regression_selection.json').read_text()))
print('Training warnings:', json.loads((RESULTS / 'regression_info.json').read_text())['warnings'])


## Clustering
Only development predictors are used, without outcome labels. PCA is a display of full-space clustering; silhouette excludes DBSCAN noise. Clusters are not diagnoses.

In [ ]:
display(pd.read_csv(RESULTS / 'clustering_metrics.csv'))
display(Image(filename=str(RESULTS / 'clustering.png')))
display(json.loads((RESULTS / 'clustering_info.json').read_text()))


## Course coverage

In [ ]:
display(Markdown(COURSE_COVERAGE))

## Generated methods and results report

In [ ]:
display(Markdown((ROOT / 'REPORT.md').read_text(encoding='utf-8-sig')))

## Verification
These are the active data-contract and modelling checks, retained here instead of separate test files.

In [ ]:
"""Checks for material leakage, splitting and preprocessing invariants."""
import sys
import unittest
from pathlib import Path
import numpy as np

class DataContractTests(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.raw, cls.X, cls.y, cls.eligible, cls.reg = load_data()

    def test_one_row_per_patient(self):
        self.assertEqual(len(self.raw), self.raw.Serial.nunique())

    def test_no_diagnostic_or_followup_predictors(self):
        self.assertEqual(list(self.X.columns), FEATURES)
        self.assertFalse(set(FEATURES) & {'Serial', 'Hospital', 'Sample', 'Year', 'Month', 'ICU', 'OPD', 'Final', 'Platelet2'})
        self.assertFalse(any(('PCR' in c or 'MAT' in c or c[-1:].isdigit() for c in FEATURES)))
        self.assertTrue(set(SYMPTOMS).issubset(FEATURES))

    def test_neutral_labels_and_admission_cohort(self):
        self.assertTrue(self.y.isin([0, 1]).all())
        self.assertTrue((self.y == 1).equals(self.raw.Final.astype(int) == 1))
        self.assertTrue(self.X.loc[self.eligible, SYMPTOMS].notna().any(axis=1).all())

    def test_sentinels_and_categories(self):
        for col in CATEGORICAL:
            self.assertTrue(set(self.X[col].dropna().unique()).issubset({'1', '2'}))
        self.assertTrue(self.reg[self.raw.Platelet2 == '99'].isna().all())

    def test_saved_split_and_source_match(self):
        import joblib
        b = joblib.load(RESULTS / 'classification_bundle.joblib')
        self.assertEqual(b['source_hash'], data_hash())
        self.assertFalse(set(b['X_train'].index) & set(b['X_test'].index))
        self.assertEqual(len(b['X_train']) + len(b['X_test']), int(self.eligible.sum()))
        for model in b['models'].values():
            p = model.predict_proba(b['X_test'])
            self.assertTrue(np.isfinite(p).all())
            np.testing.assert_allclose(p.sum(axis=1), 1)
        row = b['X_test'].iloc[[0]].copy()
        for col in row.columns:
            row[col] = np.nan
        self.assertTrue(np.isfinite(b['models']['Random Forest'].predict_proba(row)).all())


"""Check leakage boundaries and development-only selection in the expanded study."""
import json
import hashlib
import sys
import unittest
from pathlib import Path
import numpy as np
import pandas as pd

class ModellingTests(unittest.TestCase):

    def test_preprocessing_never_learns_validation_values(self):
        _, X, _, eligible, _ = load_data()
        development = X.loc[eligible].iloc[:60].copy()
        development[NUMERIC] = 10.0
        validation = X.loc[eligible].iloc[60:70].copy()
        validation[NUMERIC] = 100000.0
        prepare = preprocessor().fit(development)
        imputer = prepare.named_transformers_['numeric'].named_steps['impute']
        before = imputer.statistics_.copy()
        prepare.transform(validation)
        np.testing.assert_array_equal(before, imputer.statistics_)
        np.testing.assert_allclose(before, 10.0)

    def test_basis_expands_continuous_variables_only(self):
        X = np.array([[1, 2, 3, 4, 0, 1], [2, 3, 4, 5, 1, 0]])
        result = NumericPolynomialBasis().fit_transform(X)
        self.assertEqual(result.shape, (2, 16))
        np.testing.assert_array_equal(result[:, -2:], X[:, -2:])

    def test_clustering_does_not_score_noise_as_a_cluster(self):
        Z = np.array([[0, 0], [0.1, 0], [10, 10], [10.1, 10], [100, 100]])
        self.assertIsNone(clustering_summary(Z, np.full(5, -1))['silhouette'])
        self.assertIsNone(clustering_summary(Z, np.zeros(5))['silhouette'])
        summary = clustering_summary(Z, np.array([0, 0, 1, 1, -1]))
        self.assertEqual(summary['noise_fraction'], 0.2)
        self.assertEqual(summary['clusters'], 2)
        self.assertGreater(summary['silhouette'], 0.9)

    def test_saved_selections_match_development_scores(self):
        cv = pd.read_csv(RESULTS / 'classification_cv.csv')
        configs = json.loads((RESULTS / 'selected_configs.json').read_text())
        expected = {'Linear SVM', 'RBF SVM', 'Decision tree (Gini)', 'Decision tree (entropy)', 'Bagging', 'AdaBoost', 'Gradient boosting', 'KNN', 'Soft voting', 'LightGBM', 'CatBoost', 'Random Forest', 'XGBoost', 'Logistic regression'}
        self.assertEqual(set(configs), expected)
        for name, config in configs.items():
            scores = cv[cv.model == name].groupby('candidate').roc_auc.mean()
            self.assertEqual(config['candidate'], int(scores.idxmax()))
            self.assertAlmostEqual(config['cv_mean_auc'], scores.max())
            self.assertEqual(cv[(cv.model == name) & (cv.candidate == config['candidate'])].fold.nunique(), 5)
        chosen = json.loads((RESULTS / 'model_selection.json').read_text())
        self.assertFalse(chosen['test_used_for_selection'])
        self.assertEqual(chosen['selected_model'], max(configs, key=lambda n: configs[n]['cv_mean_auc']))
        reg = pd.read_csv(RESULTS / 'regression_cv.csv')
        reg_configs = json.loads((RESULTS / 'regression_selected_configs.json').read_text())
        for name, config in reg_configs.items():
            scores = reg[reg.model == name].groupby('candidate').rmse.mean()
            self.assertEqual(config['candidate'], int(scores.idxmin()))
        info = json.loads((RESULTS / 'clustering_info.json').read_text())
        self.assertFalse(info['uses_target_labels'])
        self.assertFalse(info['uses_test_patients'])

    def test_completed_artifacts_match_recorded_lineage(self):
        manifest = json.loads((RESULTS / 'run_manifest.json').read_text())
        for name, expected in manifest['artifact_sha256'].items():
            self.assertEqual(hashlib.sha256((RESULTS / name).read_bytes()).hexdigest(), expected, name)
        import joblib
        bundle = joblib.load(RESULTS / 'regression_bundle.joblib')
        for name, predicted in bundle['predictions'].items():
            self.assertEqual(len(predicted), len(bundle['actual']), name)
            self.assertTrue(np.isfinite(predicted).all(), name)

def verify_project():
    suite = unittest.TestSuite()
    for case in [DataContractTests, ModellingTests]:
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError('Project verification failed.')
    return result


In [ ]:
verification = verify_project()

## Optional dashboard
The complete dashboard source is embedded here. Its runner is generated in the temporary working directory, so no app.py or supporting modules are needed. Launch from the final cell or use the parent PowerShell launcher.

In [ ]:
DASHBOARD_SOURCE = '"""Local academic dashboard. Launch using ../run_dashboard.ps1."""\nimport json\nfrom pathlib import Path\nimport joblib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nfrom sklearn.metrics import confusion_matrix, balanced_accuracy_score, precision_score, recall_score\nfrom sklearn.tree import plot_tree\nst.set_page_config(page_title=\'Sri Lankan Patient Data | RF Lab\', page_icon=\'🌿\', layout=\'wide\')\nst.markdown(\'<style>\\n.block-container {padding-top:2rem;max-width:1450px}\\nh1 {color:#153f46} h2,h3 {color:#23626c}\\n[data-testid="stMetric"] {background:#eef6f6;border:1px solid #d9e7e7;border-radius:12px;padding:16px}\\n</style>\', unsafe_allow_html=True)\nst.title(\'Applied ML modelling lab\')\nst.caption(\'Sri Lankan leptospirosis study · Admission features · Reproducible model comparison\')\nst.warning(NOTICE)\nif not (RESULTS / \'data_audit.json\').exists():\n    st.info(\'Run the training script first: ../train_project.ps1\')\n    st.stop()\n\ndef read_json(name):\n    return json.loads((RESULTS / name).read_text(encoding=\'utf-8\'))\n\n@st.cache_resource\ndef bundles(fingerprint):\n    return (joblib.load(RESULTS / \'classification_bundle.joblib\'), joblib.load(RESULTS / \'regression_bundle.joblib\'))\naudit = read_json(\'data_audit.json\')\nif audit[\'sha256\'] != data_hash():\n    st.error(\'The source CSV has changed. Retrain before inspecting results or making predictions.\')\n    st.stop()\nbundle, reg_bundle = bundles(audit[\'sha256\'] + str((RESULTS / \'classification_bundle.joblib\').stat().st_mtime_ns) + str((RESULTS / \'regression_bundle.joblib\').stat().st_mtime_ns))\nmetrics = pd.read_csv(RESULTS / \'classification_metrics.csv\')\nwith st.sidebar:\n    st.header(\'Experiment\')\n    st.write(\'Source: 2016–2019 hospital study\')\n    st.write(\'Target: Final code 1 vs code 2\')\n    st.write(\'Split: 80% development / 20% test\')\n    st.write(\'Random seed: 42\')\n    st.caption(\'Models selected with development cross-validation. Dashboard controls do not change the published benchmark.\')\n    st.download_button(\'Download project report\', (ROOT / \'REPORT.md\').read_text(encoding=\'utf-8\'), file_name=\'leptospirosis_report.md\', mime=\'text/markdown\')\ncards = st.columns(4)\nfor c, label, value in zip(cards, [\'Source patients\', \'Eligible patients\', \'Admission predictors\', \'Held-out patients\'], [audit[\'rows\'], audit[\'classification_eligible\'], len(FEATURES), audit[\'classification_test_n\']]):\n    c.metric(label, value)\ntabs = st.tabs([\'Data quality\', \'Model comparison\', \'Explore RF\', \'Regression\', \'Course coverage\', \'Clustering\'])\nwith tabs[0]:\n    left, right = st.columns([1, 1.6])\n    with left:\n        st.subheader(\'What one row represents\')\n        st.write(\'One patient. Serial is checked for uniqueness and excluded from predictors.\')\n        st.write(f"The source contains {audit[\'columns\']} columns. An explicit allowlist retains {len(FEATURES)} predictors.")\n        st.write(f"Excluded from the primary cohort: {audit[\'excluded_no_admission_symptoms\']} patients without usable admission symptom data.")\n        counts = pd.DataFrame({\'Final code\': list(audit[\'source_final_counts\']), \'Patients\': list(audit[\'source_final_counts\'].values())})\n        st.dataframe(counts, hide_index=True)\n        st.info(\'99 and blanks are provisionally treated as missing. The codebook must verify this convention, especially where 99 could also be a real numeric measurement.\')\n    with right:\n        missing = pd.read_csv(RESULTS / \'feature_audit.csv\').sort_values(\'missing_fraction\', ascending=False)\n        st.subheader(\'Missingness in the eligible cohort\')\n        st.bar_chart(missing.head(20).set_index(\'feature\')[\'missing_fraction\'], horizontal=True)\n    st.dataframe(missing, hide_index=True, width=\'stretch\')\n    st.download_button(\'Download all-column selection manifest\', (RESULTS / \'column_manifest.csv\').read_bytes(), file_name=\'column_manifest.csv\', mime=\'text/csv\')\nwith tabs[1]:\n    st.subheader(\'Held-out classification benchmark\')\n    if (RESULTS / \'model_selection.json\').exists():\n        selection = read_json(\'model_selection.json\')\n        st.info(f"Selected using development folds: {selection[\'selected_model\']} (mean ROC AUC {selection[\'cv_mean_auc\']:.3f}). Test results below do not select the model.")\n    st.caption(\'Class metrics refer to Final source code 1. Threshold = 0.50. Higher is better except Brier score.\')\n    st.dataframe(metrics.style.format({c: \'{:.3f}\' for c in metrics.columns if c != \'model\'}), hide_index=True, width=\'stretch\')\n    st.image(str(RESULTS / \'classification_curves.png\'), width=\'stretch\')\n    cv = pd.read_csv(RESULTS / \'classification_cv.csv\')\n    st.subheader(\'Development cross-validation\')\n    summary = cv.groupby([\'model\', \'candidate\']).roc_auc.agg([\'mean\', \'std\']).reset_index()\n    st.dataframe(summary, hide_index=True)\n    gaps = cv.groupby([\'model\', \'candidate\'])[[\'train_roc_auc\', \'roc_auc\']].mean().reset_index() if \'train_roc_auc\' in cv else None\n    if gaps is not None:\n        with st.expander(\'Training versus validation: overfitting check\'):\n            gaps[\'train_validation_gap\'] = gaps.train_roc_auc - gaps.roc_auc\n            st.dataframe(gaps, hide_index=True, width=\'stretch\')\n    with st.expander(\'Inspect a classifier confusion matrix\'):\n        chosen = st.selectbox(\'Classification model\', list(bundle[\'models\']), key=\'comparison_model\')\n        cm = read_json(\'classification_curves.json\')[chosen][\'confusion_matrix\']\n        st.dataframe(pd.DataFrame(cm, index=[\'Actual code 2\', \'Actual code 1\'], columns=[\'Predicted code 2\', \'Predicted code 1\']))\n        st.caption(\'Code 1 is the numerical positive class: top row TN / FP; bottom row FN / TP. Threshold = 0.50.\')\n        if chosen.startswith(\'Decision tree\'):\n            selected_tree = bundle[\'models\'][chosen]\n            fig, ax = plt.subplots(figsize=(18, 7))\n            plot_tree(selected_tree.named_steps[\'model\'], feature_names=selected_tree.named_steps[\'prepare\'].get_feature_names_out(), class_names=[\'Code 2\', \'Code 1\'], max_depth=3, filled=True, fontsize=7, ax=ax)\n            st.pyplot(fig)\n            plt.close(fig)\n            st.caption(\'Selected tree shown to depth 3. Impurity criterion and pruning parameters appear below.\')\n    with st.expander(\'Selected parameters and uncertainty\'):\n        st.json(read_json(\'selected_configs.json\'))\n        st.write(\'500-resample percentile intervals on the fixed test split. Not a significance test between algorithms.\')\n        st.json(read_json(\'classification_intervals.json\'))\n    with st.expander(\'Hospital-holdout sensitivity\'):\n        st.write(\'Largest hospital held out, using predeclared parameters selected without its records. Hospital itself is not a predictor. This analysis checks one source of dataset shift.\')\n        st.json(read_json(\'hospital_sensitivity.json\'))\n    st.download_button(\'Download classification metrics\', (RESULTS / \'classification_metrics.csv\').read_bytes(), file_name=\'classification_metrics.csv\', mime=\'text/csv\')\nwith tabs[2]:\n    st.subheader(\'Explore the fitted Random Forest\')\n    st.caption(\'Exploration of frozen test outputs. Changing the threshold here does not tune or replace the reported model.\')\n    threshold = st.slider(\'Exploratory class-1 probability threshold\', 0.05, 0.95, 0.5, 0.05)\n    p = bundle[\'probabilities\'][\'Random Forest\']\n    y = bundle[\'y_test\']\n    pred = p >= threshold\n    columns = st.columns(3)\n    columns[0].metric(\'Balanced accuracy\', f\'{balanced_accuracy_score(y, pred):.3f}\')\n    columns[1].metric(\'Code-1 recall\', f\'{recall_score(y, pred, zero_division=0):.3f}\')\n    columns[2].metric(\'Code-1 precision\', f\'{precision_score(y, pred, zero_division=0):.3f}\')\n    cm = confusion_matrix(y, pred, labels=[0, 1])\n    st.dataframe(pd.DataFrame(cm, index=[\'Actual code 2\', \'Actual code 1\'], columns=[\'Predicted code 2\', \'Predicted code 1\']), width=\'stretch\')\n    importance = pd.read_csv(RESULTS / \'rf_importance.csv\').head(15)\n    st.subheader(\'Permutation importance\')\n    st.caption(\'Mean decrease in held-out ROC AUC. Descriptive, not causal; correlated features may share importance.\')\n    st.bar_chart(importance.set_index(\'feature\')[\'mean_auc_drop\'], horizontal=True)\n    with st.expander(\'Inspect an individual tree\'):\n        model = bundle[\'models\'][\'Random Forest\']\n        tree_index = st.number_input(\'Tree index\', min_value=0, max_value=len(model.named_steps[\'model\'].estimators_) - 1, value=0)\n        fig, ax = plt.subplots(figsize=(18, 8))\n        plot_tree(model.named_steps[\'model\'].estimators_[tree_index], feature_names=model.named_steps[\'prepare\'].get_feature_names_out(), class_names=[\'Code 2\', \'Code 1\'], max_depth=2, filled=True, fontsize=8, ax=ax)\n        st.pyplot(fig)\n        plt.close(fig)\n        st.caption(\'Truncated to depth 2 for legibility. A single tree is not the whole forest.\')\n    with st.expander(\'Try a hypothetical record\'):\n        st.write(\'Inputs use source categories 1 and 2. Blank numeric values and Missing categories are imputed. This predicts a source code, not a verified diagnosis.\')\n        labels = {\'Feverad\': \'Fever\', \'Headachead\': \'Headache\', \'Musclepainad\': \'Muscle pain\', \'Cnsuffusionad\': \'Conjunctival suffusion\', \'Jaundicead\': \'Jaundice\', \'OliguriaAd\': \'Oliguria\', \'SOBadd\': \'Shortness of breath\', \'Nauseaad\': \'Nausea\', \'Floodexposure\': \'Flood exposure\', \'Paddyfieldexposure\': \'Paddy-field exposure\'}\n        with st.form(\'hypothetical\'):\n            row = {c: np.nan for c in FEATURES}\n            columns = st.columns(3)\n            for i, (col, label, maxval) in enumerate([(\'Age\', \'Age\', 110), (\'PRad\', \'Admission pulse rate\', 250), (\'SBPadd\', \'Admission systolic pressure\', 300), (\'DBPadd\', \'Admission diastolic pressure\', 200)]):\n                value = columns[i % 3].number_input(label, min_value=0, max_value=maxval, value=0, help=\'0 and 99 mean unknown under the provisional data contract; out-of-range values are also imputed.\')\n                bounds = {\'Age\': (0, 110), \'PRad\': (20, 250), \'SBPadd\': (40, 300), \'DBPadd\': (20, 200)}\n                lo, hi = bounds[col]\n                row[col] = float(value) if value and value != 99 and (lo <= value <= hi) else np.nan\n            sex = columns[1].selectbox(\'Sex source code\', [\'Missing\', \'1\', \'2\'])\n            row[\'Sex\'] = np.nan if sex == \'Missing\' else sex\n            for i, (col, label) in enumerate(labels.items()):\n                value = columns[i % 3].selectbox(label + \' (source code)\', [\'Missing\', \'1\', \'2\'])\n                row[col] = np.nan if value == \'Missing\' else value\n            submit = st.form_submit_button(\'Predict source code\')\n        if submit:\n            if all((pd.isna(row[c]) for c in SYMPTOMS)):\n                st.error(\'Enter at least one admission symptom code, as required by the training cohort.\')\n            else:\n                q = float(bundle[\'models\'][\'Random Forest\'].predict_proba(pd.DataFrame([row])[FEATURES])[0, 1])\n                st.metric(\'Uncalibrated probability of Final code 1\', f\'{q:.1%}\')\n                st.caption(\'Hypothetical inputs may be outside the training distribution. Do not use this as a clinical decision.\')\nwith tabs[3]:\n    st.subheader(\'Continuous-outcome regression comparison\')\n    info = read_json(\'regression_info.json\')\n    st.write(f"Target: Platelet2. Eligible patients: {info[\'eligible_n\']}; development: {info[\'train_n\']}; test: {info[\'test_n\']}.")\n    st.info(\'Day-two timing is inferred from the suffix and source count units are unverified. Follow-up availability selects a small subgroup; this experiment is exploratory.\')\n    st.write(\'RF and ANN make out-of-fold predictions on development patients. A ridge regressor combines those predictions. Held-out patients are used only for evaluation.\')\n    r = pd.read_csv(RESULTS / \'regression_metrics.csv\')\n    st.dataframe(r.style.format({c: \'{:.3f}\' for c in r.columns if c != \'model\'}), hide_index=True, width=\'stretch\')\n    st.caption(\'Lower MAE/RMSE is better. Negative R² indicates performance below a test-mean reference.\')\n    if (RESULTS / \'regression_selection.json\').exists():\n        st.info(\'Selected using development RMSE: \' + read_json(\'regression_selection.json\')[\'selected_model\'])\n        with st.expander(\'Regression candidates and regularization\'):\n            st.dataframe(pd.read_csv(RESULTS / \'regression_cv.csv\'), hide_index=True)\n            st.json(read_json(\'regression_selected_configs.json\'))\n    st.image(str(RESULTS / \'hybrid_regression.png\'), width=\'stretch\')\n    if info[\'warnings\']:\n        st.warning(\'Training emitted warnings; see details before interpreting ANN results.\')\n        with st.expander(\'Training warnings\'):\n            st.write(info[\'warnings\'])\n    st.download_button(\'Download regression metrics\', (RESULTS / \'regression_metrics.csv\').read_bytes(), file_name=\'regression_metrics.csv\', mime=\'text/csv\')\nwith tabs[4]:\n    st.markdown((ROOT / \'COURSE_COVERAGE.md\').read_text(encoding=\'utf-8\'))\n    if (RESULTS / \'run_manifest.json\').exists():\n        with st.expander(\'Local experiment lineage\'):\n            st.json(read_json(\'run_manifest.json\'))\n    with st.expander(\'Full methods and results report\'):\n        st.markdown((ROOT / \'REPORT.md\').read_text(encoding=\'utf-8\'))\nwith tabs[5]:\n    st.subheader(\'Unsupervised exploration of development patients\')\n    if (RESULTS / \'clustering_metrics.csv\').exists():\n        st.info(\'Clustering uses development predictors only, without outcome labels. Clusters are not diagnoses.\')\n        st.dataframe(pd.read_csv(RESULTS / \'clustering_metrics.csv\'), hide_index=True, width=\'stretch\')\n        st.caption(\'Silhouette uses full prepared features and excludes DBSCAN noise. Compare noise fraction and cluster sizes alongside silhouette; blank scores mean fewer than two usable clusters.\')\n        st.image(str(RESULTS / \'clustering.png\'), width=\'stretch\')\n        st.json(read_json(\'clustering_info.json\'))\n        st.download_button(\'Download clustering metrics\', (RESULTS / \'clustering_metrics.csv\').read_bytes(), file_name=\'clustering_metrics.csv\', mime=\'text/csv\')\n    else:\n        st.info(\'Rerun training to generate the course clustering experiments.\')\n'

def export_dashboard():
    """Generate a temporary self-contained Streamlit runner from this notebook."""
    notebook = json.loads(NOTEBOOK.read_text(encoding='utf-8-sig'))
    core = '\n\n'.join(''.join(cell['source']) for cell in notebook['cells']
        if cell['cell_type'] == 'code' and 'core' in cell.get('metadata', {}).get('tags', []))
    runtime = ROOT / 'dashboard.py'
    prefix = 'import os\nimport matplotlib\nmatplotlib.use("Agg")\nos.environ["LEPTO_PROJECT"] = '+repr(str(PROJECT))+'\nos.environ["LEPTO_RUNTIME_ROOT"] = '+repr(str(ROOT))+'\n'
    runtime.write_text(prefix+core+'\n'+DASHBOARD_SOURCE, encoding='utf-8')
    return runtime

def launch_dashboard(wait=False, port=8501):
    runtime = export_dashboard()
    command = [sys.executable, '-m', 'streamlit', 'run', str(runtime),
        '--server.address', '127.0.0.1', '--server.port', str(port),
        '--browser.gatherUsageStats', 'false']
    if wait:
        subprocess.run(command, check=True)
    else:
        process = subprocess.Popen(command, creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0))
        print(f'Dashboard: http://localhost:{port} (process {process.pid})')
        return process


In [ ]:
START_DASHBOARD = False
if START_DASHBOARD:
    dashboard_process = launch_dashboard()
# To stop a dashboard launched above: dashboard_process.terminate()
